# 01 · Data Cleaning & Preparation

**Project:** E-Commerce Sales & Customer Behavior Analysis (Olist, Brazil)

**Goal of this notebook:** load the 9 raw Olist tables, find data-quality problems, fix them, and produce
clean analysis-ready tables that every other notebook, the SQL work, and the Power BI dashboard use.

**Outputs (saved to `data/processed/`):**
- `fact_orders.csv` – one row per order with revenue, payment, delivery, and review fields
- `fact_order_items.csv` – one row per item sold, with English product category and seller info
- `data_quality_log.csv` – every issue found and the decision taken

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path('../data/raw')
OUT = Path('../data/processed')
OUT.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:,.2f}'.format)

## 1. Load the raw data

Two loading decisions matter here:
- **Zip codes are read as text.** Brazilian zip prefixes can start with 0 (e.g. `01151` in São Paulo).
  Reading them as numbers would silently turn `01151` into `1151`.
- **The translation file starts with a hidden byte-order mark (BOM)**, so it is read with `utf-8-sig`;
  otherwise the first column name would be corrupted and the join would fail.

In [2]:
order_dates = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
               'order_delivered_customer_date', 'order_estimated_delivery_date']

orders = pd.read_csv(RAW / 'olist_orders_dataset.csv', parse_dates=order_dates)
customers = pd.read_csv(RAW / 'olist_customers_dataset.csv', dtype={'customer_zip_code_prefix': str})
items = pd.read_csv(RAW / 'olist_order_items_dataset.csv', parse_dates=['shipping_limit_date'])
payments = pd.read_csv(RAW / 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(RAW / 'olist_order_reviews_dataset.csv',
                      parse_dates=['review_creation_date', 'review_answer_timestamp'])
products = pd.read_csv(RAW / 'olist_products_dataset.csv')
sellers = pd.read_csv(RAW / 'olist_sellers_dataset.csv', dtype={'seller_zip_code_prefix': str})
translation = pd.read_csv(RAW / 'product_category_name_translation.csv', encoding='utf-8-sig')

tables = {'orders': orders, 'customers': customers, 'order_items': items, 'payments': payments,
          'reviews': reviews, 'products': products, 'sellers': sellers, 'translation': translation}

overview = pd.DataFrame({
    name: {'rows': len(df), 'columns': df.shape[1],
           'missing_cells': int(df.isna().sum().sum()),
           'duplicate_rows': int(df.duplicated().sum())}
    for name, df in tables.items()
}).T
overview

,rows,columns,missing_cells,duplicate_rows
orders,99441,8,4908,0
customers,99441,5,0,0
order_items,112650,7,0,0
payments,103886,5,0,0
reviews,99224,7,145903,0
products,32951,9,2448,0
sellers,3095,4,0,0
translation,71,2,0,0


The geolocation table (1M rows of zip-code coordinates) is not used: every question in this project
is answered at the state level, which the customers and sellers tables already contain.

## 2. Data-quality checks

Each finding is added to a log so the decisions are documented and reproducible.

In [3]:
issues = []
def log(table, issue, rows_affected, decision):
    issues.append({'table': table, 'issue': issue, 'rows_affected': rows_affected, 'decision': decision})

### 2.1 `customer_id` is not a real customer
In this dataset a new `customer_id` is generated for **every order**. The real person is identified by
`customer_unique_id`. Using the wrong column would make it look like nobody ever buys twice.

In [4]:
n_ids = customers['customer_id'].nunique()
n_people = customers['customer_unique_id'].nunique()
print(f'customer_id values:        {n_ids:,}')
print(f'customer_unique_id values: {n_people:,}')
print(f'Difference (repeat buyers hidden by customer_id): {n_ids - n_people:,}')
log('customers', 'customer_id is generated per order, not per person', n_ids - n_people,
    'Use customer_unique_id for all customer-level analysis')

customer_id values:        99,441
customer_unique_id values: 96,096
Difference (repeat buyers hidden by customer_id): 3,345


### 2.2 Order status and orders with no items

In [5]:
print(orders['order_status'].value_counts())

no_items = orders[~orders['order_id'].isin(items['order_id'])]
print(f'\nOrders with no item rows: {len(no_items):,}')
print(no_items['order_status'].value_counts())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Orders with no item rows: 775
order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64


In [6]:
log('orders', 'Orders with no item rows (mostly unavailable / canceled)', len(no_items),
    'Excluded from revenue; kept in fact_orders with order_value = 0')

delivered_no_date = orders[(orders['order_status'] == 'delivered') &
                           orders['order_delivered_customer_date'].isna()]
log('orders', "Status 'delivered' but no delivery date", len(delivered_no_date),
    'Kept for revenue, excluded from delivery-time metrics')

carrier_before_purchase = orders[orders['order_delivered_carrier_date'] < orders['order_purchase_timestamp']]
log('orders', 'Carrier pickup date earlier than purchase date (timestamp error)', len(carrier_before_purchase),
    'Carrier date not used in metrics; delivery time measured purchase -> customer delivery')

print(f"Delivered without a delivery date: {len(delivered_no_date)}")
print(f"Carrier date before purchase date: {len(carrier_before_purchase)}")

Delivered without a delivery date: 8
Carrier date before purchase date: 166


### 2.3 Incomplete months at the start and end of the data
The platform was ramping up in late 2016 and the data export stops in October 2018. Including these
months would distort trends and month-over-month growth.

In [7]:
monthly_counts = orders.groupby(orders['order_purchase_timestamp'].dt.to_period('M')).size()
print(monthly_counts.to_string())

order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M


In [8]:
sparse_months = monthly_counts[(monthly_counts.index < pd.Period('2017-01', 'M')) |
                               (monthly_counts.index > pd.Period('2018-08', 'M'))]
log('orders', 'Sparse months: Sep-Dec 2016 (launch) and Sep-Oct 2018 (data cut-off)', int(sparse_months.sum()),
    'Trend analysis limited to Jan 2017 - Aug 2018 (20 full months)')

### 2.4 Products: missing categories, untranslated categories, misspelled columns

In [9]:
missing_cat = products['product_category_name'].isna().sum()
untranslated = sorted(set(products['product_category_name'].dropna()) - set(translation['product_category_name']))
print(f'Products with no category: {missing_cat}')
print(f'Categories missing from the translation table: {untranslated}')
print([c for c in products.columns if 'lenght' in c])

Products with no category: 610
Categories missing from the translation table: ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']
['product_name_lenght', 'product_description_lenght']


In [10]:
log('products', 'Missing product category', int(missing_cat), "Labelled 'unknown'")
log('translation', 'Categories with no English translation', len(untranslated),
    'Translated manually (pc_gamer, portable_kitchen_food_processors)')
log('products', "Misspelled columns ('lenght')", 2, 'Renamed to product_name_length / product_description_length')

### 2.5 Reviews: some orders have more than one review

In [11]:
reviews_per_order = reviews.groupby('order_id').size()
multi = (reviews_per_order > 1).sum()
print(f'Orders with more than one review: {multi:,}')
print(f'Orders with no review: {(~orders["order_id"].isin(reviews["order_id"])).sum():,}')
log('reviews', 'Orders with more than one review', int(multi),
    'Keep the most recently answered review per order')

Orders with more than one review: 547


Orders with no review: 768


### 2.6 Payments: undefined types, zero values, and multiple payment rows per order

In [12]:
print(payments['payment_type'].value_counts())
print(f"\nZero-value payments: {(payments['payment_value'] == 0).sum()}")
print(f"Payments with 0 installments: {(payments['payment_installments'] == 0).sum()}")
print(f"Orders paid with more than one payment row: {(payments.groupby('order_id').size() > 1).sum():,}")
print(f"Orders with no payment record: {(~orders['order_id'].isin(payments['order_id'])).sum()}")

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

Zero-value payments: 9
Payments with 0 installments: 2


Orders paid with more than one payment row: 2,961
Orders with no payment record: 1


In [13]:
log('payments', "payment_type = 'not_defined'", int((payments['payment_type'] == 'not_defined').sum()),
    "Relabelled 'unknown'")
log('payments', 'payment_installments = 0', int((payments['payment_installments'] == 0).sum()),
    'Set to 1 (a single payment)')
log('payments', 'Orders split across several payment rows (e.g. card + voucher)',
    int((payments.groupby('order_id').size() > 1).sum()),
    'Aggregated to one row per order; main payment type = the one with the largest value')

### 2.7 Do payments match item prices?
Revenue can be measured from items (price + freight) or from payments. They should be close.

In [14]:
item_totals = items.groupby('order_id').apply(lambda g: (g['price'] + g['freight_value']).sum(), include_groups=False)
pay_totals = payments.groupby('order_id')['payment_value'].sum()
diff = (pay_totals - item_totals).dropna()
print(f'Orders compared: {len(diff):,}')
print(f'Orders where payment and item total differ by more than R$1: {(diff.abs() > 1).sum():,} '
      f'({(diff.abs() > 1).mean():.2%})')

Orders compared: 98,665
Orders where payment and item total differ by more than R$1: 249 (0.25%)


In [15]:
log('payments', 'Payment total differs from item total by > R$1 (installment interest, vouchers)',
    int((diff.abs() > 1).sum()),
    'Revenue measured from items (price + freight) - consistent with category-level analysis')

## 3. Cleaning

In [16]:
# --- Products: translate categories, fill missing, fix column names ---
manual_translation = pd.DataFrame({
    'product_category_name': ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'],
    'product_category_name_english': ['pc_gamer', 'portable_kitchen_food_processors'],
})
translation_full = pd.concat([translation, manual_translation], ignore_index=True)

products_clean = (products
    .rename(columns={'product_name_lenght': 'product_name_length',
                     'product_description_lenght': 'product_description_length'})
    .merge(translation_full, on='product_category_name', how='left'))
products_clean['category'] = products_clean['product_category_name_english'].fillna('unknown')

# --- Reviews: one review per order (latest answer) ---
reviews_clean = (reviews
    .sort_values('review_answer_timestamp')
    .drop_duplicates('order_id', keep='last')
    [['order_id', 'review_score', 'review_creation_date']])

# --- Payments: one row per order ---
payments_clean = payments.copy()
payments_clean['payment_type'] = payments_clean['payment_type'].replace('not_defined', 'unknown')
payments_clean['payment_installments'] = payments_clean['payment_installments'].clip(lower=1)

main_type = (payments_clean.sort_values('payment_value', ascending=False)
             .drop_duplicates('order_id')[['order_id', 'payment_type']])
payments_order = (payments_clean.groupby('order_id')
                  .agg(payment_value=('payment_value', 'sum'),
                       payment_installments=('payment_installments', 'max'),
                       payment_methods_used=('payment_type', 'nunique'))
                  .reset_index()
                  .merge(main_type, on='order_id'))

# --- Items: one row per order ---
items_order = (items.groupby('order_id')
               .agg(item_count=('order_item_id', 'count'),
                    seller_count=('seller_id', 'nunique'),
                    product_value=('price', 'sum'),
                    freight_value=('freight_value', 'sum'))
               .reset_index())
items_order['order_value'] = items_order['product_value'] + items_order['freight_value']

### Build `fact_orders` (one row per order)

**Metric definitions used across the whole project:**
- **Revenue (order value)** = product price + freight, from the order items table, for **delivered** orders.
- **Delivery days** = time from purchase to delivery to the customer.
- **Late** = delivered on a calendar day after the estimated delivery date shown to the customer
  (`is_late` is stored as 1/0 so its average is the late-delivery rate).

In [17]:
fact_orders = (orders
    .merge(customers, on='customer_id', how='left')
    .merge(items_order, on='order_id', how='left')
    .merge(payments_order, on='order_id', how='left')
    .merge(reviews_clean[['order_id', 'review_score']], on='order_id', how='left'))

for col in ['item_count', 'seller_count', 'product_value', 'freight_value', 'order_value']:
    fact_orders[col] = fact_orders[col].fillna(0)

fact_orders['is_delivered'] = fact_orders['order_status'].eq('delivered')
fact_orders['purchase_date'] = fact_orders['order_purchase_timestamp'].dt.date
fact_orders['purchase_month'] = fact_orders['order_purchase_timestamp'].dt.to_period('M').astype(str)
fact_orders['in_trend_window'] = fact_orders['purchase_month'].between('2017-01', '2018-08')

has_delivery = fact_orders['is_delivered'] & fact_orders['order_delivered_customer_date'].notna()
delivery_days = (fact_orders['order_delivered_customer_date'] - fact_orders['order_purchase_timestamp']).dt.total_seconds() / 86400
delay_days = (fact_orders['order_delivered_customer_date'].dt.normalize() - fact_orders['order_estimated_delivery_date']).dt.days

fact_orders['delivery_days'] = delivery_days.where(has_delivery).round(2)
fact_orders['delay_days'] = delay_days.where(has_delivery)
fact_orders['is_late'] = (fact_orders['delay_days'] > 0).astype(float).where(has_delivery)   # 1 = late, 0 = on time

status_labels = ['On time', '1-3 days late', '4-7 days late', '8-14 days late', '15+ days late']
fact_orders['delivery_status'] = pd.cut(fact_orders['delay_days'], bins=[-np.inf, 0, 3, 7, 14, np.inf],
                                        labels=status_labels).astype(object)
fact_orders['delivery_status_order'] = fact_orders['delivery_status'].map({l: i for i, l in enumerate(status_labels, 1)}).astype('Int64')

fact_orders = fact_orders[[
    'order_id', 'customer_id', 'customer_unique_id', 'customer_city', 'customer_state', 'order_status',
    'is_delivered', 'order_purchase_timestamp', 'purchase_date', 'purchase_month', 'in_trend_window',
    'order_approved_at', 'order_delivered_customer_date', 'order_estimated_delivery_date',
    'item_count', 'seller_count', 'product_value', 'freight_value', 'order_value',
    'payment_value', 'payment_type', 'payment_installments', 'payment_methods_used',
    'review_score', 'delivery_days', 'delay_days', 'is_late', 'delivery_status', 'delivery_status_order']]
fact_orders.head(3).T

,0,1,2
order_id,e481f51cbdc54678b7cc49136f2d6af7,53cdb2fc8bc7dce0b6741e2150273451,47770eb9100c2d0c44946d9cf07ec65d
customer_id,9ef432eb6251297304e76186b10a928d,b0830fb4747a6c6d20dea0b8c802d7ef,41ce2a54c0b03bf3443c3d931a367089
customer_unique_id,7c396fd4830fd04220f754e42b4e5bff,af07308b275d755c9edb36a90c618231,3a653a41f6f9fc3d2a113cf8398680e8
customer_city,sao paulo,barreiras,vianopolis
customer_state,SP,BA,GO
order_status,delivered,delivered,delivered
is_delivered,True,True,True
order_purchase_timestamp,2017-10-02 10:56:33,2018-07-24 20:41:37,2018-08-08 08:38:49
purchase_date,2017-10-02,2018-07-24,2018-08-08
purchase_month,2017-10,2018-07,2018-08


### Build `fact_order_items` (one row per item sold)

In [18]:
fact_order_items = (items
    .merge(products_clean[['product_id', 'category', 'product_weight_g']], on='product_id', how='left')
    .merge(sellers[['seller_id', 'seller_city', 'seller_state']], on='seller_id', how='left')
    .merge(fact_orders[['order_id', 'order_status', 'is_delivered', 'order_purchase_timestamp',
                        'purchase_month', 'customer_unique_id', 'customer_state', 'review_score']],
           on='order_id', how='left'))
fact_order_items['item_value'] = fact_order_items['price'] + fact_order_items['freight_value']
fact_order_items.head(3).T

,0,1,2
order_id,00010242fe8c5a6d1ba2dd792cb16214,00018f77f2f0320c557190d7a144bdd3,000229ec398224ef6ca0657da4fc703e
order_item_id,1,1,1
product_id,4244733e06e7ecb4970a6e2683c13e61,e5f2d52b802189ee658865ca93d83a8f,c777355d18b72b67abbeef9df44fd0fd
seller_id,48436dade18ac8b2bce089ec2a041202,dd7ddc04e1b6c2c614352b383efe2d36,5b51032eddd242adc84c38acab88f23d
shipping_limit_date,2017-09-19 09:45:35,2017-05-03 11:05:13,2018-01-18 14:48:30
price,58.90,239.90,199.00
freight_value,13.29,19.93,17.87
category,cool_stuff,pet_shop,furniture_decor
product_weight_g,650.00,"30,000.00","3,050.00"
seller_city,volta redonda,sao paulo,borda da mata


## 4. Validation
These checks fail loudly if a join duplicated or lost rows.

In [19]:
assert fact_orders['order_id'].is_unique, 'fact_orders must have one row per order'
assert len(fact_orders) == len(orders), 'no orders lost or duplicated'
assert len(fact_order_items) == len(items), 'no items lost or duplicated'
assert np.isclose(fact_orders['order_value'].sum(), (items['price'] + items['freight_value']).sum())
assert fact_orders['customer_unique_id'].notna().all(), 'every order has a customer'
assert fact_order_items['category'].notna().all(), 'every item has a category'
assert fact_orders['review_score'].dropna().between(1, 5).all()
print('All validation checks passed.')

All validation checks passed.


In [20]:
delivered = fact_orders[fact_orders['is_delivered']]
print(f"Delivered orders:     {len(delivered):,}")
print(f"Unique customers:     {delivered['customer_unique_id'].nunique():,}")
print(f"Revenue (R$):         {delivered['order_value'].sum():,.2f}")
print(f"Average order value:  R$ {delivered['order_value'].mean():,.2f}")

Delivered orders:     96,478
Unique customers:     93,358
Revenue (R$):         15,419,773.75
Average order value:  R$ 159.83


## 5. Save outputs

In [21]:
fact_orders.to_csv(OUT / 'fact_orders.csv', index=False)
fact_order_items.to_csv(OUT / 'fact_order_items.csv', index=False)

quality_log = pd.DataFrame(issues)
quality_log.to_csv(OUT / 'data_quality_log.csv', index=False)
quality_log

,table,issue,rows_affected,decision
0,customers,"customer_id is generated per order, not per pe...",3345,Use customer_unique_id for all customer-level ...
1,orders,Orders with no item rows (mostly unavailable /...,775,Excluded from revenue; kept in fact_orders wit...
2,orders,Status 'delivered' but no delivery date,8,"Kept for revenue, excluded from delivery-time ..."
3,orders,Carrier pickup date earlier than purchase date...,166,Carrier date not used in metrics; delivery tim...
4,orders,Sparse months: Sep-Dec 2016 (launch) and Sep-O...,349,Trend analysis limited to Jan 2017 - Aug 2018 ...
5,products,Missing product category,610,Labelled 'unknown'
6,translation,Categories with no English translation,2,"Translated manually (pc_gamer, portable_kitche..."
7,products,Misspelled columns ('lenght'),2,Renamed to product_name_length / product_descr...
8,reviews,Orders with more than one review,547,Keep the most recently answered review per order
9,payments,payment_type = 'not_defined',3,Relabelled 'unknown'


**Next:** `02_exploratory_analysis.ipynb` uses these clean tables to answer the business questions.